# Peningkatan IndoBERT — **IndoBERTweet** + weighted loss  (Colab/GPU)

Track **C**. Dua perbaikan utama vs notebook baseline:
1. **Ganti model → `indolem/indobertweet-base-uncased`** — dilatih pada **Twitter
   Indonesia** (bahasa medsos/alay), domainnya jauh lebih cocok dgn komentar YouTube
   daripada indobert-base (Wikipedia/berita). *Kandidat lompatan terbesar.*
2. **Weighted cross-entropy loss** — bobot kelas terbalik thd frekuensi → bantu kelas
   minoritas (Netral), berguna untuk versi imbalanced (v1/v3).
3. Anti-overfit: early stopping + `load_best_model_at_end`.

Set **`VERSION_FLAG`** (samakan dgn SVM) & **`MODEL_NAME`**. Runtime → GPU.

## 0. Dependency + GPU

In [ ]:
%pip install -q "transformers>=4.40" "torch" "pymongo[srv]" dnspython certifi scikit-learn matplotlib pandas numpy
import torch; print("CUDA:",torch.cuda.is_available())

## 1. Pilih versi + model, baca `processed_bert`

In [ ]:
import os, pandas as pd
from pymongo import MongoClient
import certifi
from getpass import getpass
from sklearn.model_selection import train_test_split

VERSION_FLAG = "in_balanced_set"   # in_set6k | in_balanced_set | in_set10k | in_balanced10k
MODEL_NAME   = "indolem/indobertweet-base-uncased"   # pembanding: "indobenchmark/indobert-base-p1"

MONGO_URI=os.environ.get("MONGO_URI","") or getpass("MONGO_URI: ")
DB=os.environ.get("MONGO_DB_NAME","youtube_sentiment")
client=MongoClient(MONGO_URI,tlsCAFile=certifi.where(),serverSelectionTimeoutMS=20000); client.admin.command("ping")
LABELS=["Negatif","Netral","Positif"]; LABEL2ID={l:i for i,l in enumerate(LABELS)}
df=pd.DataFrame(list(client[DB]["processed_bert"].find({VERSION_FLAG:True},{"_id":0,"comment_id":1,"bert":1,"label":1})))
df["label_id"]=df["label"].map(LABEL2ID); df=df.sort_values("comment_id").reset_index(drop=True)
tmp,df_test=train_test_split(df,test_size=0.10,stratify=df["label_id"],random_state=42)
df_train,df_val=train_test_split(tmp,test_size=0.20/0.90,stratify=tmp["label_id"],random_state=42)
print(f"{MODEL_NAME} | versi={VERSION_FLAG} | train={len(df_train)} val={len(df_val)} test={len(df_test)}")

## 2. Tokenizer + Dataset

In [ ]:
from transformers import AutoTokenizer
import torch
MAX_LEN=128; SEED=42
tok=AutoTokenizer.from_pretrained(MODEL_NAME)
class DS(torch.utils.data.Dataset):
    def __init__(self,texts,labels):
        self.enc=tok(list(texts),truncation=True,max_length=MAX_LEN,padding=True); self.labels=list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self,i):
        it={k:torch.tensor(v[i]) for k,v in self.enc.items()}; it["labels"]=torch.tensor(self.labels[i]); return it
ds_train=DS(df_train["bert"].astype(str),df_train["label_id"])
ds_val=DS(df_val["bert"].astype(str),df_val["label_id"])
ds_test=DS(df_test["bert"].astype(str),df_test["label_id"])

## 3. Model + weighted loss + TrainingArguments

In [ ]:
import numpy as np
from collections import Counter
from transformers import (AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          EarlyStoppingCallback, set_seed)
from sklearn.metrics import f1_score, accuracy_score
set_seed(SEED)
model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=3,
    id2label={i:l for i,l in enumerate(LABELS)},label2id=LABEL2ID)

# bobot kelas = terbalik thd frekuensi (membantu kelas minoritas/Netral)
cnt=Counter(df_train["label_id"]); n=len(df_train)
class_w=torch.tensor([n/(3*cnt[i]) for i in range(3)],dtype=torch.float)
print("class weights:",class_w.tolist())

class WeightedTrainer(Trainer):
    def compute_loss(self,model,inputs,return_outputs=False,**kw):
        labels=inputs.pop("labels"); out=model(**inputs)
        loss=torch.nn.functional.cross_entropy(out.logits,labels,weight=class_w.to(out.logits.device))
        return (loss,out) if return_outputs else loss

def compute_metrics(p):
    pr=np.argmax(p.predictions,axis=1)
    return {"macro_f1":f1_score(p.label_ids,pr,average="macro"),"accuracy":accuracy_score(p.label_ids,pr)}

args=TrainingArguments(output_dir="out",num_train_epochs=8,per_device_train_batch_size=16,
    per_device_eval_batch_size=32,learning_rate=2e-5,weight_decay=0.01,warmup_ratio=0.1,
    eval_strategy="epoch",save_strategy="epoch",load_best_model_at_end=True,
    metric_for_best_model="macro_f1",greater_is_better=True,seed=SEED,logging_steps=50,report_to="none")
trainer=WeightedTrainer(model=model,args=args,train_dataset=ds_train,eval_dataset=ds_val,
    compute_metrics=compute_metrics,callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

## 4. Fine-tune

In [ ]:
trainer.train(); print(trainer.evaluate())

## 5. Evaluasi test + simpan

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
pred=trainer.predict(ds_test); yp=np.argmax(pred.predictions,axis=1).tolist(); yt=df_test["label_id"].tolist()
ids=list(range(3)); rep=classification_report(yt,yp,labels=ids,target_names=LABELS,output_dict=True,zero_division=0)
m={"accuracy":round(accuracy_score(yt,yp),4),"macro_f1":round(f1_score(yt,yp,average="macro"),4),
   "per_class":{l:{"f1-score":round(rep[l]["f1-score"],4),"support":int(rep[l]["support"])} for l in LABELS},
   "confusion_matrix":confusion_matrix(yt,yp,labels=ids).tolist(),"labels":LABELS}
print("IndoBERTweet —",VERSION_FLAG,"| macro-F1",m["macro_f1"],"acc",m["accuracy"])
import json
SUF={"in_set6k":"is6","in_balanced_set":"ibs","in_set10k":"is10","in_balanced10k":"ib10"}[VERSION_FLAG]
json.dump({"model":MODEL_NAME,"version":VERSION_FLAG,"test":m},open(f"indobertweet_metrics_{SUF}.json","w"),ensure_ascii=False,indent=2)
print(f"Disimpan: indobertweet_metrics_{SUF}.json  (download -> outputs/reports/)")

**Lanjut:** jalankan untuk tiap `VERSION_FLAG`, download `indobertweet_metrics_*.json`
ke `outputs/reports/`. Bandingkan dgn IndoBERT baseline & SVM. Untuk pembanding adil,
jalankan juga dengan `MODEL_NAME="indobenchmark/indobert-base-p1"` (efek weighted loss saja).